# Actividad 8: Sistema de Monitorización Proactiva y Gobernanza Operativa de IA
## Caso de Estudio: MexBan - Inferencia de DistilBERT en Producción

**Estudiante:** Juan Manuel Carreón Chávez  
**Matrícula:** AL07182807  
**Materia:** Gestión de Proyectos de IA  
**Fecha:** 16 de Noviembre de 2026  

---
### 1. Resumen y Objetivos
En este notebook se implementa la arquitectura integral de **observabilidad moderna** (Métricas, Logs Estructurados y Trazas Distribuidas OpenTelemetry) para el modelo de análisis de sentimientos **DistilBERT**, seleccionado en la Actividad 4 y puesto en servicio en producción para la institución bancaria **MexBan**.

Se abordan los siguientes componentes clave:
1. **Telemetría y Pilares de Observabilidad:** Captura de métricas Golden Signals (latencia percentiles, throughput, errores), logs JSON estructurados con enmascaramiento PII y trazas de spans.
2. **Detección Estadística de Data Drift y Concept Drift:** Implementación de Population Stability Index (PSI), prueba Kolmogorov-Smirnov (KS-test), Distancia de Wasserstein y tasa Out-Of-Vocabulary (OOV).
3. **Motor de Alertas Inteligentes (SLO & Error Budgets):** Definición rigurosa de SLIs/SLOs, cálculo de tasa de quemado (burn rate) y reducción de fatiga con histéresis y persistencia temporal.
4. **Simulación de Incidentes Críticos y Runbooks Automatizados:**
   - *Incidente 1:* Saturación de concurrencia y pico de latencia (Runbook RB-01: Auto-escalado HPA de 2 a 6 réplicas y activación de caché Redis).
   - *Incidente 2:* Data Drift severo y degradación de F1 por lanzamiento de Cripto-Billetera (Runbook RB-02: Fallback seguro, reentrenamiento automatizado con MLflow y Canary deployment de v2.0).
5. **Seguimiento con MLflow:** Registro de experimentos, parámetros, métricas continuas, artefactos y versionado en MLflow Model Registry.

In [1]:
# Configuración de entorno e importaciones
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Agregar directorio raíz
sys.path.insert(0, os.path.abspath('..'))

from src.metrics_logger import ObservabilityTelemetry
from src.drift_detector import DriftDetector
from src.alert_engine import SmartAlertEngine
from src.runbook_automation import RunbookAutomationEngine
from src.observability_service import MexBanObservabilityService

print('Entorno y módulos de observabilidad cargados exitosamente.')

### 2. Definición de Indicadores: Logs Estructurados y Trazas Distribuidas
A continuación inspeccionamos el esquema JSON estructurado generado para cada inferencia, garantizando anonimización PII y trazabilidad extremo a extremo mediante IDs de spans compatibles con OpenTelemetry.

In [2]:
telemetry = ObservabilityTelemetry()
sample_log = telemetry.record_inference(
    text='Excelente atención en la sucursal Reforma, trámite de tarjeta muy rápido',
    prediction='Positivo',
    confidence=0.962,
    probabilities={'Positivo': 0.962, 'Negativo': 0.018, 'Neutro': 0.020},
    latency_ms=17.4,
    channel='mobile_banking',
    http_status=200
)
print(telemetry.export_sample_json_log())

### 3. Ejecución de la Simulación Integral de Producción y Registro en MLflow
Ejecutamos el ciclo continuo de simulación a lo largo de 26 ventanas operativas que abarcan operación normal, los dos incidentes simulados y la intervención automatizada de los Runbooks.

In [3]:
from scripts.run_full_simulation import run_simulation
df_sim, alerts_df, incidents_list = run_simulation()
print(f'Total de registros simulados: {len(df_sim)}')
df_sim.head()

### 4. Visualización del Dashboard Operativo en Tiempo Real
Mostramos los 4 paneles de control del estado operativo del modelo (Golden Signals y Salud de ML).

In [4]:
from IPython.display import Image, display
display(Image(filename='../dashboards/img1_realtime_operational_dashboard.png'))

### 5. Análisis del Incidente 1: Degradación de Latencia y Auto-escalado HPA
Revisión de la curva de latencia p95, saturación de CPU y mitigación ejecutada por el Runbook RB-01.

In [5]:
display(Image(filename='../dashboards/img2_incident1_latency_mitigation.png'))

### 6. Análisis del Incidente 2: Data Drift y Reentrenamiento MLflow v2.0
Evaluación de la deriva de datos (PSI) tras la introducción del vocabulario de cripto-billetera, degradación de F1-Score y recuperación mediante despliegue de DistilBERT v2.0.

In [6]:
display(Image(filename='../dashboards/img3_incident2_datadrift_retraining.png'))

### 7. Gobernanza en MLflow Model Registry
Inspección del estado de los modelos versionados en MLflow.

In [7]:
display(Image(filename='../dashboards/img5_mlflow_tracking_ui_mockup.png'))